# Bolton Businesses Without a Website
This notebook finds small businesses in **Bolton, England** that have no website listed on OpenStreetMap.

**Just press the ▶ button on each cell in order, or go to Runtime → Run all.**

In [ ]:
import requests, csv, json, time, io
from collections import Counter
from IPython.display import display, FileLink
import pandas as pd
print('Libraries ready.')

In [ ]:
# Bolton Metropolitan Borough bounding box (south, west, north, east)
BOLTON_BBOX = (53.490, -2.590, 53.670, -2.310)

OVERPASS_URL = 'https://overpass-api.de/api/interpreter'

BUSINESS_TAGS = ['shop', 'amenity', 'office', 'craft', 'leisure', 'tourism']

NON_BUSINESS_AMENITIES = {
    'parking', 'bicycle_parking', 'bench', 'waste_basket', 'post_box',
    'telephone', 'toilets', 'drinking_water', 'fountain', 'recycling',
    'shelter', 'bus_station', 'ferry_terminal', 'taxi', 'car_sharing',
    'charging_station', 'fuel', 'atm', 'vending_machine', 'school',
    'kindergarten', 'college', 'university', 'library', 'place_of_worship',
    'hospital', 'clinic', 'doctors', 'dentist', 'pharmacy', 'police',
    'fire_station', 'post_office', 'community_centre', 'social_facility',
    'nursing_home', 'prison', 'courthouse', 'townhall', 'embassy',
    'grave_yard', 'crematorium',
}
print('Config set.')

In [ ]:
south, west, north, east = BOLTON_BBOX
bbox_str = f'{south},{west},{north},{east}'

tag_blocks = '\n'.join(
    f'  node["name"]["{tag}"]({bbox_str});\n  way["name"]["{tag}"]({bbox_str});'
    for tag in BUSINESS_TAGS
)
query = f'[out:json][timeout:120];\n(\n{tag_blocks}\n);\nout center;'

print('Querying Overpass API (this may take 20-40 seconds)…')
resp = requests.post(OVERPASS_URL, data={'data': query}, timeout=150)
resp.raise_for_status()
data = resp.json()
print(f'Done. Raw elements returned: {len(data["elements"])}')

In [ ]:
def has_website(tags):
    return any(tags.get(k) for k in ('website','contact:website','url','contact:url'))

def extract(element):
    tags = element.get('tags', {})
    if not tags.get('name') or has_website(tags):
        return None
    category, biz_type = None, None
    for tag in BUSINESS_TAGS:
        if tag in tags:
            category, biz_type = tag, tags[tag]
            break
    if category == 'amenity' and biz_type in NON_BUSINESS_AMENITIES:
        return None
    if element.get('type') == 'node':
        lat, lon = element.get('lat',''), element.get('lon','')
    else:
        c = element.get('center', {})
        lat, lon = c.get('lat',''), c.get('lon','')
    parts = [tags.get('addr:housenumber',''), tags.get('addr:street',''), tags.get('addr:city','') or tags.get('addr:town','')]
    return {
        'name':         tags.get('name',''),
        'business_type': biz_type or '',
        'category':     category or '',
        'address':      ', '.join(p for p in parts if p),
        'postcode':     tags.get('addr:postcode',''),
        'phone':        tags.get('phone','') or tags.get('contact:phone',''),
        'email':        tags.get('email','') or tags.get('contact:email',''),
        'opening_hours': tags.get('opening_hours',''),
        'lat': lat, 'lon': lon,
        'osm_id': element.get('id',''), 'osm_type': element.get('type',''),
    }

raw_businesses = [b for e in data['elements'] if (b := extract(e))]

# Deduplicate
seen, businesses = set(), []
for b in raw_businesses:
    key = (b['name'].lower().strip(), b['address'].lower().strip())
    if key not in seen:
        seen.add(key)
        businesses.append(b)

businesses.sort(key=lambda b: b['name'].lower())
print(f'Businesses WITHOUT a website found: {len(businesses)}')

In [ ]:
df = pd.DataFrame(businesses)

print('\nTop 20 business types:')
print(df['business_type'].value_counts().head(20).to_string())

print(f'\nWith phone:   {df["phone"].astype(bool).sum()}')
print(f'With email:   {df["email"].astype(bool).sum()}')
print(f'With address: {df["address"].astype(bool).sum()}')

print('\nFirst 10 results:')
display(df[['name','business_type','address','postcode','phone']].head(10))

In [ ]:
output_file = 'bolton_businesses_no_website.csv'
df.to_csv(output_file, index=False)
print(f'Saved {len(df)} rows to {output_file}')
print('\nClick the link below to download:')
display(FileLink(output_file))